# Giai đoạn 7: Baseline ngoài, NeuroFlowNet trên protocol sạch (GPU)

**NeuroFlowNet** (arXiv 2603.03354) là một normalizing flow có điều kiện, báo cáo r ≈ 0,50–0,54 theo protocol của họ. Notebook này **cài đặt lại theo đúng kiến trúc trong mã nguồn công khai** của họ:
- Flow đa tầng: 2 scale × 6 bước. Mỗi bước gồm một tích chập 1×1 khả nghịch và một coupling affine có điều kiện, có self-attention theo thời gian.
- Hidden 128, 8 head. Hàm mất mát là NLL; AdamW lr 1e-3; batch 64; cửa sổ 1 s, stride 20.

Mô hình được chạy trên **đúng protocol của chúng ta**. Các điều chỉnh bắt buộc, sẽ nêu rõ trong bài:
1. Một model cho mỗi bệnh nhân, train trên **cùng 20% trial calib** như mọi model khác. Dừng sớm dựa trên 20% cuối của dữ liệu calib, rồi refit trên toàn bộ calib.
2. **Target chuẩn hoá bằng thống kê của toàn bộ calib**, không dùng mean/std của chính cửa sổ test. Thống kê đó là thông tin của target, không có lúc triển khai thật.
3. Dự đoán trên các cửa sổ 1 s liên tiếp; r tính trên phần test nối liền, giống mọi model khác.

Báo cáo ba cách lấy dự đoán:

| Cách | Ý nghĩa |
|---|---|
| `nfn_mode` | Latent = 0, cho ra một dự đoán xác định |
| `nfn_sample` | Một mẫu, như cách đánh giá gốc của họ |
| `nfn_mean` | Trung bình 8 mẫu, xấp xỉ kỳ vọng có điều kiện |

Đã kiểm tra trên máy local: flow khả nghịch (sai số khoảng 1e-6), log-likelihood đúng, và toàn bộ quy trình train/dự đoán chạy được.

**Runtime: GPU.** Ước tính 5–10 phút mỗi bệnh nhân, tổng 1–2 giờ. Chạy lại được nhiều lần, tiếp tục từ bệnh nhân còn dở.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, torch
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'
RUNS_DIR = f'{ROOT}/runs'
MAIN_RUN = f'{RUNS_DIR}/shared_bip_car'
NFN_OUT  = f'{RUNS_DIR}/nfn_baseline'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ (Runtime > Change runtime type > GPU)')

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/shared.py
"""
Phase 2 - shared temporal encoder + per-subject ridge decoder (LOSO, GPU).

Evidence from phases 0-1d (15 subjects, bipolar iEEG, CAR EEG, 20 % chronological calibration):
  * the SPATIAL scalp -> contact mapping does not transfer across subjects (1a: zero-shot r ~ 0);
  * within a subject the lever is frequency-selective, long-support LINEAR temporal filtering
    (1d: random Morlet filter bank, linear, gated on top of the ridge: +0.0075, 14/15 subjects),
    while non-linearity and more random features add ~nothing at 4 min of calibration.
Hypothesis: the TEMPORAL part (which filters of the scalp signal predict intracranial activity)
is shared across subjects even though the spatial part is not. Hence:
  encoder : per-channel filter bank, shared by all channels and all subjects, trained on the
            OTHER subjects. Quadrature pairs initialised as complex Morlet 2-60 Hz (+-1.2 s
            support) + the envelope of each pair + a few free filters.
  decoder : per-subject linear readout (channels x filters -> contacts), fitted in closed form by
            ridge on the calibration trials only, exactly like the baselines.
Training is episodic so it matches deployment: each step takes a training subject, fits a ridge
readout on ~4 min of consecutive trials (support), scores Pearson r on the following trials
(query), and back-propagates through the ridge solve into the filters. Model selection: the
same procedure on 2 held-out validation subjects (first 20 % of trials -> later trials).

Evaluation on the test subject (never seen in training; same split and calibration sizes as
phases 0-1d, so ridge / base / lin_gated must reproduce the phase-1d numbers):
  ridge, base, lin_gated  baselines (lin_gated = best model of phase 1d: the bar to beat)
  enc0_gated              base + gated stage 2 on the encoder AT INITIALISATION (untrained Morlet
                          bank): control - does the gain come from learning or from the init?
  lin_ridge, enc0_ridge   single-stage controls: ridge (K-fold, shared alpha) on the random linear
                          Morlet bank / on the UNTRAINED encoder
  enc_ridge               ridge (K-fold, shared alpha) on learned encoder features alone
  enc_ridge_cal           enc_ridge with a per-contact amplitude gain fitted on out-of-fold
                          calibration predictions (same r, honest R^2)
Every model is also scored with
  r_no50     46-54 Hz removed from prediction and target (the trained filters develop a sharp 50 Hz
             peak: mains residue after the notch filter);
  r_induced  each side minus its own trial average over the test trials, i.e. how well
             the SINGLE-TRIAL (non stimulus-locked) part is predicted.
  enc_gated               base + gated stage 2 on learned encoder features      <- main model

Usage
-----
python -m s2i.shared --data DATA_DIR --out RUNS_DIR/shared_bip_car
"""
import argparse
import copy
import glob
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import StreamRidge, chrono_split, load_trials, template, tmpl_at
from s2i.metrics import per_contact_metrics
from s2i.nonlin import RandomConvFeatures, gated, summarize
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge

FS = 200
EXTRA_METRICS = ("r_no50", "r_induced")


# ----------------------------------------------------------------------------- encoder
class TemporalEncoder(nn.Module):
    """Per-channel filter bank shared by all channels and subjects: (B, C, L) -> (B, C, F, L)."""

    def __init__(self, n_pairs=12, half=240, fmin=2.0, fmax=60.0, cycles=5.0, n_free=4, envelope=True):
        super().__init__()
        self.cfg = dict(n_pairs=n_pairs, half=half, fmin=fmin, fmax=fmax, cycles=cycles, n_free=n_free,
                        envelope=envelope)
        self.n_pairs, self.envelope = n_pairs, envelope
        t = np.arange(-half, half + 1) / FS
        f = np.geomspace(fmin, fmax, n_pairs)
        gauss = np.exp(-t[None] ** 2 / (2 * (cycles / (2 * np.pi * f[:, None])) ** 2))
        w = np.zeros((2 * n_pairs + n_free, t.size))
        w[0:2 * n_pairs:2] = gauss * np.cos(2 * np.pi * f[:, None] * t[None])
        w[1:2 * n_pairs:2] = gauss * np.sin(2 * np.pi * f[:, None] * t[None])
        if n_free:
            w[2 * n_pairs, half] = 1.0                                   # identity (broadband, lag 0)
            g = np.random.default_rng(0)
            w[2 * n_pairs + 1:] = g.normal(size=(n_free - 1, t.size)) * np.exp(-t ** 2 / (2 * 0.05 ** 2))
        w /= np.linalg.norm(w, axis=1, keepdims=True)
        self.conv = nn.Conv1d(1, w.shape[0], 2 * half + 1, padding=half, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(torch.tensor(w, dtype=torch.float32)[:, None])

    @property
    def n_feat(self):
        return self.conv.out_channels + (self.n_pairs if self.envelope else 0)

    def forward(self, x):
        B, C, L = x.shape
        h = self.conv(x.reshape(B * C, 1, L))
        if self.envelope:
            p = h[:, :2 * self.n_pairs].float()
            h = torch.cat([h.float(), torch.sqrt(p[:, 0::2] ** 2 + p[:, 1::2] ** 2 + 1e-6)], 1)
        return h.reshape(B, C, -1, L)


# ----------------------------------------------------------------------------- torch helpers
def pad_trials(E, Y, idx, dev):
    L = max(E[i].shape[1] for i in idx)
    x = np.zeros((len(idx), E[idx[0]].shape[0], L), np.float32)
    y = np.zeros((len(idx), Y[idx[0]].shape[0], L), np.float32)
    m = np.zeros((len(idx), L), bool)
    for j, i in enumerate(idx):
        n = E[i].shape[1]
        x[j, :, :n], y[j, :, :n], m[j, :n] = E[i], Y[i], True
    return torch.from_numpy(x).to(dev), torch.from_numpy(y).to(dev), torch.from_numpy(m).to(dev)


def rows(enc, E, Y, idx, dev, amp, chunk=16):
    """Encoder features and targets of the given trials as (N, C*F) / (N, K) rows (padding removed)."""
    X, T = [], []
    for s in range(0, len(idx), chunk):
        x, y, m = pad_trials(E, Y, idx[s:s + chunk], dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)
        B, C, F, L = h.shape
        X.append(h.float().permute(0, 3, 1, 2).reshape(B, L, C * F)[m])
        T.append(y.permute(0, 2, 1)[m])
    return torch.cat(X), torch.cat(T)


def ridge_torch(Xs, Ys, Xq, lam):
    """Ridge on standardised features; lam is relative to the mean eigenvalue (= 1 after scaling)."""
    mu, sd = Xs.mean(0), Xs.std(0) + 1e-4
    A, Q = (Xs - mu) / sd, (Xq - mu) / sd
    ym = Ys.mean(0)
    N, D = A.shape
    G = A.T @ A / N + lam * torch.eye(D, device=A.device)
    return Q @ torch.linalg.solve(G, A.T @ (Ys - ym) / N) + ym


def col_r(P, T):
    P, T = P - P.mean(0), T - T.mean(0)
    return (P * T).sum(0) / ((P * P).sum(0).sqrt() * (T * T).sum(0).sqrt() + 1e-8)


def episode(n_trials, mean_len, rng, sup_sec, n_query):
    """Chronological episode: ~sup_sec of consecutive support trials, gap 1, then n_query trials."""
    n_q = min(n_query, max(1, n_trials // 4))
    n_s = int(np.clip(round(sup_sec * FS / mean_len), 3, n_trials - n_q - 1))
    start = int(rng.integers(0, n_trials - n_s - n_q - 1 + 1))
    return list(range(start, start + n_s)), list(range(start + n_s + 1, start + n_s + 1 + n_q))


@torch.no_grad()
def val_score(enc, lam, subjects, dev, amp, max_test):
    enc.eval()
    rs = []
    for _, E, Y in subjects:
        cal, test = chrono_split(len(E), 0.2, gap=1)
        Xs, Ys = rows(enc, E, Y, cal, dev, amp)
        Xq, Yq = rows(enc, E, Y, test[:max_test], dev, amp)
        rs.append(col_r(ridge_torch(Xs, Ys, Xq, lam), Yq).mean().item())
    return float(np.mean(rs))


# ----------------------------------------------------------------------------- training
def train_encoder(a, train, val, dev, log, ckpt_path):
    enc = TemporalEncoder(**enc_cfg(a)).to(dev)
    log_lam = nn.Parameter(torch.tensor(float(np.log(a.lam0)), device=dev))
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": a.lr, "weight_decay": 0.0},
                             {"params": [log_lam], "lr": a.lr_lam, "weight_decay": 0.0}])
    amp = a.amp and dev.type == "cuda"
    rng = np.random.default_rng(a.seed)
    mean_len = {sid: float(np.mean([e.shape[1] for e in E])) for sid, E, _ in train}
    step, best, best_state, bad, hist = 0, -1e9, None, 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
        enc.load_state_dict(ck["enc"])
        opt.load_state_dict(ck["opt"])
        with torch.no_grad():
            log_lam.copy_(ck["log_lam"])
        step, best, best_state, bad, hist = ck["step"], ck["best"], ck["best_state"], ck["bad"], ck["hist"]
        rng = np.random.default_rng(a.seed + step)
        log(f"  resumed at step {step} (best val r={best:.4f})")
    else:
        r0 = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
        hist.append((0, r0))
        log(f"  step     0 | val r (Morlet init) = {r0:.4f}")
        best, best_state = r0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
    t0, run_r = time.time(), []
    while step < a.steps and bad < a.patience:
        enc.train()
        loss = 0.0
        for si in rng.choice(len(train), a.subj_per_step, replace=len(train) < a.subj_per_step):
            sid, E, Y = train[si]
            sup, qry = episode(len(E), mean_len[sid], rng, a.sup_sec, a.n_query)
            Xs, Ys = rows(enc, E, Y, sup, dev, amp)
            Xq, Yq = rows(enc, E, Y, qry, dev, amp)
            r = col_r(ridge_torch(Xs, Ys, Xq, log_lam.exp()), Yq).mean()
            loss = loss + (1 - r) / a.subj_per_step
            run_r.append(r.item())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
        opt.step()
        step += 1
        if step % a.eval_every == 0 or step == a.steps:
            vr = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
            hist.append((step, vr))
            if vr > best:
                best, bad, best_state = vr, 0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
            else:
                bad += 1
            log(f"  step {step:5d} | train query r={np.mean(run_r):.4f} | val r={vr:.4f} | "
                f"lam={log_lam.exp().item():.3g} | {time.time() - t0:.0f}s" + (" *" if bad == 0 else ""))
            run_r, t0 = [], time.time()
            if ckpt_path:
                torch.save({"enc": enc.state_dict(), "opt": opt.state_dict(), "log_lam": log_lam.detach(),
                            "step": step, "best": best, "best_state": best_state, "bad": bad, "hist": hist},
                           ckpt_path + ".tmp")
                os.replace(ckpt_path + ".tmp", ckpt_path)
    enc.load_state_dict(best_state[0])
    return enc, best_state[1], best, hist


def enc_cfg(a):
    return dict(n_pairs=a.n_pairs, half=a.half, fmin=a.fmin, fmax=a.fmax, cycles=a.cycles,
                n_free=a.n_free, envelope=not a.no_env)


# ----------------------------------------------------------------------------- evaluation
def np_feat(enc, dev, amp):
    """Encoder as a numpy feature function (C, L) -> (C*F, L) for the StreamRidge machinery."""
    enc = enc.eval()

    @torch.no_grad()
    def f(e):
        x = torch.from_numpy(np.ascontiguousarray(e, np.float32))[None].to(dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)[0]
        return h.float().reshape(-1, h.shape[-1]).cpu().numpy()
    return f


_LINE = butter(4, (46, 54), btype="bandstop", fs=FS, output="sos")


def r_without_line(P, T):
    """Per-contact r after removing 46-54 Hz (mains residue) from BOTH prediction and target."""
    fp = [sosfiltfilt(_LINE, p, axis=1) for p in P]
    ft = [sosfiltfilt(_LINE, t, axis=1) for t in T]
    return per_contact_metrics(fp, ft, FS, bands=False)["r"]


def r_single_trial(P, T):
    """
    Per-contact r of the single-trial parts: each side minus ITS OWN trial average over the test
    trials (prediction minus the mean prediction, target minus the mean target). Removing the same
    template from both sides would add a shared component and inflate r.
    """
    tP, tT = template(P), template(T)
    return per_contact_metrics([p - tmpl_at(tP, p.shape[1]) for p in P],
                               [t - tmpl_at(tT, t.shape[1]) for t in T], FS, bands=False)["r"]


def amplitude_calibrate(kf, Yc, P):
    """
    Per-contact least-squares gain on the OUT-OF-FOLD calibration predictions. Alpha is chosen by
    r, which ignores amplitude; this fixes the amplitude (R^2) without changing r (gain >= 0).
    """
    ym = kf.s["ym"][:, None]
    num = sum(((o - ym) * (y - ym)).sum(1) for o, y in zip(kf.oof, Yc))
    den = sum(((o - ym) ** 2).sum(1) for o in kf.oof)
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)[:, None]
    return [(ym + g * (p - ym)).astype(np.float32) for p in P]


def evaluate(E, Y, enc0, enc, a, dev, log):
    amp = a.amp and dev.type == "cuda"
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    f0, f1 = np_feat(enc0, dev, amp), np_feat(enc, dev, amp)
    lin = RandomConvFeatures(E[0].shape[0], M=512, seed=0, mode="linear")
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            m["r_no50"] = r_without_line(P, Yt)
            m["r_induced"] = r_single_trial(P, Yt)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]],
                         **{x: [float(v) for v in m[x]] for x in EXTRA_METRICS}}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        if not a.fast_eval:
            for name, feat in [("lin_gated", lin), ("enc0_gated", f0), ("enc_gated", f1)]:
                P, on[name] = gated(base, Ec, Yc, Et, feat)
                score(name, P)
        for name, feat in [("lin_ridge", lin)] * (not a.fast_eval) + [("enc0_ridge", f0)]:
            score(name, KFoldRidge(feat, ALPHAS_WIDE, shared=True, keep_oof=False).fit(Ec, Yc).predict(Et))
        kf = KFoldRidge(f1, ALPHAS_WIDE, shared=True).fit(Ec, Yc)
        P = kf.predict(Et)
        score("enc_ridge", P)
        score("enc_ridge_cal", amplitude_calibrate(kf, Yc, P))
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  **{x: {k: v[x] for k, v in out.items()} for x in EXTRA_METRICS},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


KEY_TESTS = [("enc_ridge", "enc0_ridge", "learning helps (vs same encoder untrained)?"),
             ("enc_ridge", "lin_ridge", "learned filters beat random filters?"),
             ("enc0_ridge", "lin_ridge", "Morlet init beats random filters?"),
             ("enc_ridge", "base", "beats the strong linear baseline?"),
             ("lin_ridge", "base", "single-stage filter bank beats LAG9?")]


def key_tests(results, out_dir, log):
    """Paired subject-level tests for the decisive comparisons, on r and on every EXTRA_METRICS."""
    from scipy.stats import wilcoxon
    lines, js = ["\n### key comparisons (paired over subjects)",
                 "| calib | question | A | B | metric | mean A | mean B | A - B | A wins | Wilcoxon p |",
                 "|---|---|---|---|---|---|---|---|---|---|"], []
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"] and "r_no50" in r["fracs"][f]]
        metrics = ("r",) + tuple(x for x in EXTRA_METRICS if rows and x in rows[0]["fracs"][f])
        for ma, mb, q in KEY_TESTS:
            if not rows or ma not in rows[0]["fracs"][f]["mean"] or mb not in rows[0]["fracs"][f]["mean"]:
                continue
            for metric in metrics:
                if metric == "r":
                    get = lambda r, m: r["fracs"][f]["mean"][m]                         # noqa: E731
                else:
                    get = lambda r, m: float(np.nanmean(r["fracs"][f][metric][m]))     # noqa: E731
                x = np.array([get(r, ma) for r in rows])
                y = np.array([get(r, mb) for r in rows])
                try:
                    p = float(wilcoxon(x, y).pvalue)
                except ValueError:
                    p = float("nan")
                js.append({"frac": f, "a": ma, "b": mb, "metric": metric, "mean_a": float(x.mean()),
                           "mean_b": float(y.mean()), "diff": float((x - y).mean()), "wins": int((x > y).sum()),
                           "n": len(x), "p": p})
                lines.append(f"| {f} | {q} | {ma} | {mb} | {metric} | {x.mean():.3f} | {y.mean():.3f} | "
                             f"{(x - y).mean():+.4f} | {(x > y).sum()}/{len(x)} | {p:.4f} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "summary.md"), "a", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "key_tests.json"), "w") as fh:
        json.dump(js, fh, indent=1)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None, help="subset of subjects to load")
    ap.add_argument("--folds", nargs="*", default=None, help="test subjects to run (default: all)")
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.1, 0.2])
    # encoder
    ap.add_argument("--n_pairs", type=int, default=12)
    ap.add_argument("--half", type=int, default=240, help="filter half-length in samples (240 = 1.2 s)")
    ap.add_argument("--fmin", type=float, default=2.0)
    ap.add_argument("--fmax", type=float, default=60.0)
    ap.add_argument("--cycles", type=float, default=5.0)
    ap.add_argument("--n_free", type=int, default=4)
    ap.add_argument("--no_env", action="store_true")
    # episodic training
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--eval_every", type=int, default=150)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--subj_per_step", type=int, default=2)
    ap.add_argument("--sup_sec", type=float, default=240.0, help="support size (seconds), ~4 min as at test")
    ap.add_argument("--n_query", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--lr_lam", type=float, default=1e-2)
    ap.add_argument("--lam0", type=float, default=1.0)
    ap.add_argument("--n_val_subj", type=int, default=2)
    ap.add_argument("--val_trials", type=int, default=40, help="test trials per validation subject")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--fast_eval", action="store_true",
                    help="only ridge, base, enc0_ridge, enc_ridge, enc_ridge_cal (for seed / ablation runs)")
    ap.add_argument("--reeval", action="store_true",
                    help="redo the evaluation of finished folds with their saved encoders (no retraining)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    a.amp = not a.no_amp
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "shared.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    subjects = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"skip {sid}: {len(E)} trials, {s.n_targets} targets")
            continue
        subjects.append((sid, E, Y))
        log(f"{sid}: {len(E)} trials, {E[0].shape[0]} scalp, {s.n_targets} targets")
    folds = [s for s in subjects if not a.folds or s[0] in a.folds]

    results = []
    for fi, (sid, E, Y) in enumerate(folds):
        fp = os.path.join(a.out, f"fold_{sid}.json")
        if os.path.exists(fp):
            done = json.load(open(fp))
            # --reeval redoes only folds evaluated by an older version (missing a metric), so it can resume
            if not a.reeval or all(x in fr for fr in done["fracs"].values() for x in EXTRA_METRICS):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
        torch.manual_seed(a.seed + fi)
        log("=" * 78)
        log(f"[{fi + 1}/{len(folds)}] TEST {sid}")
        pool = [s for s in subjects if s[0] != sid]
        order = np.random.default_rng(a.seed + fi).permutation(len(pool))
        val = [pool[i] for i in order[:a.n_val_subj]]
        train = [pool[i] for i in order[a.n_val_subj:]]
        log(f"  train={[s[0] for s in train]} val={[s[0] for s in val]}")
        t0 = time.time()
        enc_path = os.path.join(a.out, f"enc_{sid}.pt")
        if os.path.exists(enc_path):
            ck = torch.load(enc_path, map_location=dev, weights_only=False)
            enc = TemporalEncoder(**ck["cfg"]).to(dev)
            enc.load_state_dict(ck["state"])
            lam, best, hist = ck["lam"], ck["best_val"], ck["hist"]
            log(f"  loaded trained encoder (val r={best:.4f})")
        else:
            ckpt = os.path.join(a.out, f"ckpt_{sid}.pt")
            enc, lam, best, hist = train_encoder(a, train, val, dev, log, ckpt)
            torch.save({"state": enc.state_dict(), "cfg": enc.cfg, "lam": lam, "best_val": best, "hist": hist},
                       enc_path)
            if os.path.exists(ckpt):
                os.remove(ckpt)
        enc0 = TemporalEncoder(**enc.cfg).to(dev)
        log(f"  encoder: val r {hist[0][1]:.4f} (init) -> {best:.4f} (trained), lam={lam:.3g}, "
            f"{enc.n_feat} features/channel | {time.time() - t0:.0f}s")
        r = evaluate(E, Y, enc0, enc, a, dev, log)
        r.update(subject=sid, val_subjects=[s[0] for s in val], val_r_init=hist[0][1], val_r=best, lam=lam,
                 minutes=(time.time() - t0) / 60)
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
        del enc, enc0
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    if results:
        summarize(results, a.out, log)
        key_tests(results, a.out, log)
        vi = [r["val_r_init"] for r in results]
        vt = [r["val_r"] for r in results]
        log(f"validation subjects, ridge readout on 20 %: Morlet init r={np.mean(vi):.4f} -> trained {np.mean(vt):.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/nfn_baseline.py
"""
External baseline: NeuroFlowNet (conditional normalizing flow, arXiv 2603.03354) re-implemented from the
architecture of its public code (github.com/hdy6438/NeuroFlowNet) and run under OUR clean protocol.

Architecture (as in the public code): MultiScaleConditionalFlow - n_scales scales of n_steps flow steps;
each step = invertible 1x1 convolution (QR init) + conditional affine coupling whose network is
conv(k3) -> ReLU -> conv(k1) -> ReLU -> multi-head self-attention over time -> conv(k3) -> (shift, log-scale),
shift = 10 tanh(.), log-scale clamped to [-5, 5]; the scalp EEG window is concatenated to the coupling
input at every step; standard-normal prior; trained by negative log-likelihood (AdamW, lr 1e-3,
weight decay 1e-2, batch 64, grad-norm clip 1); 1-s windows, training stride 20 samples, scalp windows
z-scored per window and channel.

Changes required by the clean protocol (stated in the paper):
  * one model per patient trained on the SAME calibration trials as every other model (first 20 % of the
    trials, chronological); early stopping on the last 20 % of those calibration trials (patience 10,
    criterion = our r), then refit on all calibration trials for the selected number of epochs;
  * the iEEG target is normalised with the per-channel mean / std of the CALIBRATION data, not with the
    statistics of each test window (which is target information and unavailable at deployment);
  * test trials are predicted with consecutive 1-s windows (overlaps averaged); r per contact on the
    concatenated test trials, as for all models.
  * the number of output channels must be divisible by 2^n_scales; missing channels are padded with
    independent noise channels that are ignored.
Predictions reported: nfn_mode (latent = 0), nfn_sample (one sample, as in the original evaluation) and
nfn_mean (average of 8 samples ~ conditional mean).

Usage
-----
python -m s2i.nfn_baseline --data DATA_DIR --out RUNS_DIR/nfn_baseline
"""
import argparse
import copy
import glob
import json
import math
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn

from s2i.ceiling import chrono_split
from s2i.metrics import per_contact_metrics
from s2i.shared import r_single_trial

FS, WIN = 200, 200


# ----------------------------------------------------------------------------- flow
class Coupling(nn.Module):
    def __init__(self, c, hid, ctx, heads):
        super().__init__()
        self.h = c // 2
        self.c1 = nn.Conv1d(self.h + ctx, hid, 3, padding=1)
        self.c2 = nn.Conv1d(hid, hid, 1)
        self.ln = nn.LayerNorm(hid)
        self.att = nn.MultiheadAttention(hid, heads, batch_first=True)
        self.c3 = nn.Conv1d(hid, 2 * (c - self.h), 3, padding=1)
        nn.init.zeros_(self.c3.weight)
        nn.init.zeros_(self.c3.bias)

    def st(self, xa, ctx):
        h = Fn.relu(self.c2(Fn.relu(self.c1(torch.cat([xa, ctx], 1)))))
        t = h.transpose(1, 2)
        q = self.ln(t)
        t = t + self.att(q, q, q, need_weights=False)[0]
        shift, logs = self.c3(t.transpose(1, 2)).chunk(2, 1)
        return 10 * torch.tanh(shift / 10), logs.clamp(-5, 5)

    def forward(self, x, ctx):
        xa, xb = x[:, :self.h], x[:, self.h:]
        shift, logs = self.st(xa, ctx)
        return torch.cat([xa, xb * logs.exp() + shift], 1), logs.sum((1, 2))

    def inverse(self, y, ctx):
        ya, yb = y[:, :self.h], y[:, self.h:]
        shift, logs = self.st(ya, ctx)
        return torch.cat([ya, (yb - shift) * (-logs).exp()], 1)


class Step(nn.Module):
    def __init__(self, c, hid, ctx, heads):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(c, c))
        self.W = nn.Parameter(q)
        self.cp = Coupling(c, hid, ctx, heads)

    def forward(self, x, ctx):
        x = torch.einsum("ij,bjl->bil", self.W, x)
        ld = x.shape[2] * torch.linalg.slogdet(self.W)[1]
        x, ld2 = self.cp(x, ctx)
        return x, ld + ld2

    def inverse(self, y, ctx):
        return torch.einsum("ij,bjl->bil", torch.linalg.inv(self.W), self.cp.inverse(y, ctx))


class Flow(nn.Module):
    def __init__(self, c_out, c_in, hid=128, n_steps=6, n_scales=2, heads=8):
        super().__init__()
        assert c_out % (2 ** n_scales) == 0
        self.scales, c = nn.ModuleList(), c_out
        for _ in range(n_scales):
            self.scales.append(nn.ModuleList([Step(c, hid, c_in, heads) for _ in range(n_steps)]))
            c //= 2
        self.c_out = c_out

    def log_prob(self, x, ctx):
        ld, zs = 0.0, []
        for steps in self.scales:
            for st in steps:
                x, l_ = st(x, ctx)
                ld = ld + l_
            half = x.shape[1] // 2
            zs.append(x[:, :half])
            x = x[:, half:]
        zs.append(x)
        prior = sum(-0.5 * (z ** 2 + math.log(2 * math.pi)).sum((1, 2)) for z in zs)
        return prior + ld

    def sample(self, ctx, temp=1.0):
        B, L = ctx.shape[0], ctx.shape[2]
        sizes, c = [], self.c_out
        for _ in self.scales:
            sizes.append(c // 2)
            c //= 2
        x = torch.randn(B, c, L, device=ctx.device) * temp
        for steps, s in zip(reversed(self.scales), reversed(sizes)):
            x = torch.cat([torch.randn(B, s, L, device=ctx.device) * temp, x], 1)
            for st in reversed(steps):
                x = st.inverse(x, ctx)
        return x


# ----------------------------------------------------------------------------- data
def load(path):
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    b = d["trial_bounds"].numpy()
    return [eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b]


def zwin(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-6)


def windows(E, Y, idx, stride, mu, sd, k_pad, rng):
    X, T = [], []
    for i in idx:
        L = E[i].shape[1]
        for s in range(0, L - WIN + 1, stride):
            X.append(zwin(E[i][:, s:s + WIN]))
            y = (Y[i][:, s:s + WIN] - mu) / sd
            if k_pad:
                y = np.concatenate([y, rng.normal(size=(k_pad, WIN)).astype(np.float32)])
            T.append(y)
    return torch.tensor(np.stack(X), dtype=torch.float32), torch.tensor(np.stack(T), dtype=torch.float32)


@torch.no_grad()
def predict(model, E, idx, mu, sd, K, dev, mode, n_mean=8, bs=64):
    """Consecutive 1-s windows per trial (last one aligned to the end, overlaps averaged)."""
    model.eval()
    out = []
    for i in idx:
        L = E[i].shape[1]
        starts = list(range(0, L - WIN + 1, WIN))
        if starts[-1] != L - WIN:
            starts.append(L - WIN)
        ctx = torch.tensor(np.stack([zwin(E[i][:, s:s + WIN]) for s in starts]), dtype=torch.float32, device=dev)
        P = []
        for b0 in range(0, len(ctx), bs):
            c = ctx[b0:b0 + bs]
            if mode == "mode":
                p = model.sample(c, temp=0.0)
            elif mode == "sample":
                p = model.sample(c)
            else:
                p = sum(model.sample(c) for _ in range(n_mean)) / n_mean
            P.append(p[:, :K].cpu().numpy())
        P = np.concatenate(P)
        acc, cnt = np.zeros((K, L)), np.zeros(L)
        for p, s in zip(P, starts):
            acc[:, s:s + WIN] += p
            cnt[s:s + WIN] += 1
        out.append((acc / cnt * sd + mu).astype(np.float32))
    return out


def r_mean(P, T):
    return float(np.nanmean(per_contact_metrics(P, T, FS, bands=False)["r"]))


# ----------------------------------------------------------------------------- per patient
def train(E, Y, fit_idx, val_idx, a, dev, log, n_epochs=None):
    K = Y[0].shape[0]
    k_pad = (-K) % (2 ** a.n_scales)
    rng = np.random.default_rng(a.seed)
    torch.manual_seed(a.seed)
    cal = np.concatenate([Y[i] for i in fit_idx], 1)
    mu, sd = cal.mean(1, keepdims=True), cal.std(1, keepdims=True) + 1e-6
    Xw, Tw = windows(E, Y, fit_idx, a.stride, mu, sd, k_pad, rng)
    model = Flow(K + k_pad, E[0].shape[0], a.hidden, a.n_steps, a.n_scales, a.heads).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-2)
    best, best_state, bad, best_ep = -1e9, None, 0, 0
    epochs = n_epochs if n_epochs is not None else a.max_epochs
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xw))[:a.steps_per_epoch * a.bs]
        tot = 0.0
        for b0 in range(0, len(perm), a.bs):
            j = perm[b0:b0 + a.bs]
            loss = -model.log_prob(Tw[j].to(dev), Xw[j].to(dev)).mean() / ((K + k_pad) * WIN)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item()
        if n_epochs is not None:
            continue
        vr = r_mean(predict(model, E, val_idx, mu, sd, K, dev, "mode"), [Y[i] for i in val_idx])
        if vr > best:
            best, bad, best_ep, best_state = vr, 0, ep + 1, copy.deepcopy(model.state_dict())
        else:
            bad += 1
        if ep % 5 == 0 or bad == 0:
            log(f"      ep {ep + 1:3d} nll/dim={tot / max(1, len(perm) // a.bs):+.3f} val r={vr:.4f}" + (" *" if bad == 0 else ""))
        if bad >= a.patience:
            break
    if n_epochs is None:
        model.load_state_dict(best_state)
    return model, mu, sd, best, best_ep


def analyze(E, Y, a, dev, log):
    n = len(E)
    cal, test = chrono_split(n, a.calib_frac, gap=1)
    n_fit = max(2, int(round(len(cal) * 0.8)))
    fit_idx, val_idx = cal[:n_fit], cal[n_fit:] or cal[-1:]
    _, _, _, best_val, best_ep = train(E, Y, fit_idx, val_idx, a, dev, log)
    log(f"    early stopping: best val r={best_val:.4f} at epoch {best_ep}; refit on all calibration trials")
    model, mu, sd, _, _ = train(E, Y, cal, None, a, dev, log, n_epochs=max(1, best_ep))
    Yt = [Y[i] for i in test]
    res = {"n_trials": n, "best_val_r": best_val, "epochs": best_ep, "models": {}}
    for mode in ("mode", "sample", "mean"):
        P = predict(model, E, test, mu, sd, Y[0].shape[0], dev, mode, a.n_mean)
        m = per_contact_metrics(P, Yt, FS, bands=False)
        ri = r_single_trial(P, Yt)
        res["models"][f"nfn_{mode}"] = {"r": float(np.nanmean(m["r"])), "r2": float(np.nanmean(m["r2"])),
                                        "r_induced": float(np.nanmean(ri)), "contacts": [float(x) for x in m["r"]]}
    log("    test: " + " ".join(f"{k} r={v['r']:.4f} r_ind={v['r_induced']:.4f}" for k, v in res["models"].items()))
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--calib_frac", type=float, default=0.2)
    ap.add_argument("--hidden", type=int, default=128)
    ap.add_argument("--n_steps", type=int, default=6)
    ap.add_argument("--n_scales", type=int, default=2)
    ap.add_argument("--heads", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--stride", type=int, default=20)
    ap.add_argument("--max_epochs", type=int, default=500)
    ap.add_argument("--steps_per_epoch", type=int, default=50)
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--n_mean", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "nfn.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    results = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        fp = os.path.join(a.out, f"nfn_{sid}.json")
        if os.path.exists(fp):
            results.append(json.load(open(fp)))
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        E, Y = load(p)
        if len(E) < 20:
            continue
        log(f"[{sid}] {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} contacts")
        r = analyze(E, Y, a, dev, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        lines = ["| model | mean r | r_induced | R² |", "|---|---|---|---|"]
        for m in results[0]["models"]:
            v = [r["models"][m] for r in results]
            lines.append(f"| {m} | {np.mean([x['r'] for x in v]):.4f} | {np.mean([x['r_induced'] for x in v]):.4f} | "
                         f"{np.mean([x['r2'] for x in v]):.4f} |")
        md = "\n".join(lines)
        log("\n" + md)
        with open(os.path.join(a.out, "nfn.md"), "w", encoding="utf-8") as fh:
            fh.write(md + "\n")


if __name__ == "__main__":
    main()

## 3. Chạy

In [ ]:
#@title 3. Train + đánh giá NeuroFlowNet (mỗi bệnh nhân một model)
!python -m s2i.nfn_baseline --data {DATA_DIR} --out {NFN_OUT}

In [ ]:
#@title 4. So sánh với model của chúng ta (calib 20%, cùng trial test)
import json, numpy as np, pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon
nfn = {json.load(open(f))['subject']: json.load(open(f)) for f in glob.glob(f'{NFN_OUT}/nfn_sub-*.json')}
fin = {json.load(open(f))['subject']: json.load(open(f)) for f in glob.glob(f'{MAIN_RUN}/readout_final/readout_sub-*.json')}
subs = sorted(set(nfn) & set(fin))
rows = []
for s in subs:
    row = {'subject': s}
    for m in ('nfn_mode', 'nfn_sample', 'nfn_mean'):
        row[m] = nfn[s]['models'][m]['r']
    for m in ('base', 'enc', 'combo'):
        row[m] = fin[s]['explicit'][m]['mean']['r']
    rows.append(row)
df = pd.DataFrame(rows).set_index('subject')
display(df.round(3)); display(df.mean().round(4).to_frame('mean r').T)
best_nfn = df[['nfn_mode', 'nfn_sample', 'nfn_mean']].mean().idxmax()
for ref in ('base', best_nfn):
    d = df['combo'] - df[ref]
    print(f'combo − {ref}: {d.mean():+.4f}, thắng {(d > 0).sum()}/{len(d)}, Wilcoxon p={wilcoxon(df["combo"], df[ref]).pvalue:.4f}')

## Cách đọc
- **`combo` > NeuroFlowNet (p < 0,05, ≥ 12/15):** claim SOTA so với một phương pháp đã công bố, chạy trên cùng protocol. Đây là bảng chính cho phần so sánh trong bài.
- **NeuroFlowNet ≈ hoặc < `base` (ridge):** một model sinh dữ liệu lớn, train trên 4 phút dữ liệu của một bệnh nhân, không thắng được ridge. Điều này khớp với phát hiện LeadFormer < ridge.
- **`nfn_sample` ≪ `nfn_mode` hoặc `nfn_mean`:** mẫu sinh ra từ model có r thấp hơn kỳ vọng có điều kiện. Đây là minh hoạ cho lập luận lý thuyết trong bài, rằng model sinh dữ liệu không thể vượt kỳ vọng có điều kiện về r.

Gửi lại bảng ở cell 4.